# wrap_model_call
## decorator

In [14]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest

load_dotenv(override=True)

model = init_chat_model(model="deepseek:deepseek-chat")

In [15]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware, wrap_tool_call


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call before <---"
    # 模型调用
    response = handler(request)
    response.result[0].content += "---> wrap_model_call after <---"

    return response


In [8]:
agent  = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [
        HumanMessage("Hello!")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hello!---> wrap_model_call before <---
================================== Ai Message ==================================

Hello! How can I assist you today?---> wrap_model_call after <---


## class

In [16]:
from langchain.agents.middleware import AgentMiddleware
class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self,
                        request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call before <---"
        # 模型调用
        response = handler(request)
        response.result[0].content += "---> wrap_model_call after <---"

        return response

In [10]:
agent  = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [
        HumanMessage("Hello!")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hello!---> wrap_model_call before <---
================================== Ai Message ==================================

Hello! How can I help you today? If you have any questions or need assistance, feel free to ask.---> wrap_model_call after <---


# wrap_tool_call
## decorator

In [17]:
from langgraph.types import Command
from typing import Any
from langchain_core.tools import tool

@tool
def get_weather(city: str, is_forecast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forecast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forecast:
        res += "明天天气也很好"

    return res

@wrap_tool_call
def wrap_tool_call_middleware(
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"Original args: {request.tool_call['args']}")
    print(f"Original result: {result}")

    request.tool_call["args"]["is_forecast"] = True
    result = handler(request)

    print(f"Modified args: {request.tool_call['args']}")
    print(f"Modified result: {result}")
    return result

agent = create_agent(
    model = model,
    tools = [get_weather],
    middleware = [wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何")]
})
for msg in response["messages"]:
    msg.pretty_print()

Original args: {'city': '北京', 'is_forecast': False}
Original result: content='北京今天天气不错' name='get_weather' tool_call_id='call_00_wsT20p1z6I2OsHOVUdXd3899'
Modified args: {'city': '北京', 'is_forecast': True}
Modified result: content='北京今天天气不错明天天气也很好' name='get_weather' tool_call_id='call_00_wsT20p1z6I2OsHOVUdXd3899'
================================ Human Message =================================

帮我查询北京今天的天气如何
================================== Ai Message ==================================

好的，我来查询北京今天的天气情况。
Tool Calls:
  get_weather (call_00_wsT20p1z6I2OsHOVUdXd3899)
 Call ID: call_00_wsT20p1z6I2OsHOVUdXd3899
  Args:
    city: 北京
    is_forecast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错明天天气也很好
================================== Ai Message ==================================

北京今天的天气**不错**！看起来是个好天气 ☀️。另外，明天的天气也很不错哦，如果您有出行计划的话，这两天都是比较适合的。


## class

In [18]:
class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self,
                       request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
            result = handler(request)
            print(f"Original args: {request.tool_call['args']}")
            print(f"Original result: {result}")

            request.tool_call["args"]["is_forecast"] = True
            result = handler(request)

            print(f"Modified args: {request.tool_call['args']}")
            print(f"Modified result: {result}")
            return result

agent = create_agent(
    model = model,
    tools = [get_weather],
    middleware = [WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何")]
})
for msg in response["messages"]:
    msg.pretty_print()

Original args: {'city': '北京', 'is_forecast': False}
Original result: content='北京今天天气不错' name='get_weather' tool_call_id='call_00_4Yf6X5SsL4Wi6hmJnijx7417'
Modified args: {'city': '北京', 'is_forecast': True}
Modified result: content='北京今天天气不错明天天气也很好' name='get_weather' tool_call_id='call_00_4Yf6X5SsL4Wi6hmJnijx7417'
================================ Human Message =================================

帮我查询北京今天的天气如何
================================== Ai Message ==================================

好的，我来查询北京今天的天气情况。
Tool Calls:
  get_weather (call_00_4Yf6X5SsL4Wi6hmJnijx7417)
 Call ID: call_00_4Yf6X5SsL4Wi6hmJnijx7417
  Args:
    city: 北京
    is_forecast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错明天天气也很好
================================== Ai Message ==================================

**北京今天的天气情况：** ☀️

今天北京的天气**不错**，很适合出门活动哦！而且明天的天气也很好。

如果你需要更具体的温度、风力等信息，可以告诉我，我再帮你进一步查询～


> before_model 执行顺序与传递顺序一致

> after_model 执行顺序与传递顺序相反

> wrap_model 包裹结构